# 🔍 Data Exploration - Retail Video Analytics

Notebook khám phá và kiểm tra data trong lakehouse (AWS S3):
- **Bronze Layer**: Raw data từ Pulsar
- **Silver Layer**: Cleaned detections
- **Gold Layer**: Aggregated metrics

---

## 1️⃣ Setup & Connection

In [2]:
import pandas as pd
import trino
from datetime import datetime, timedelta
import json

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [3]:
# Connect to Trino (gateway to Iceberg data in AWS S3)
conn = trino.dbapi.connect(
    host='localhost',
    port=8083,
    user='hungfnguyen',
    catalog='lakehouse',
    schema='rva'
)

cursor = conn.cursor()
print(f"✅ Connected to Trino")
print(f"   Catalog: {conn.catalog}")
print(f"   Schema: {conn.schema}")

✅ Connected to Trino
   Catalog: lakehouse
   Schema: rva


In [4]:
# Helper function for quick queries
def query(sql, show_query=True):
    """Execute SQL and return DataFrame"""
    if show_query:
        print(f"🔍 Query: {sql[:100]}..." if len(sql) > 100 else f"🔍 Query: {sql}")
    try:
        df = pd.read_sql(sql, conn)
        print(f"✅ Returned {len(df)} rows")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Helper function ready")

✅ Helper function ready


---
## 2️⃣ Bronze Layer Exploration

**Bronze = Raw data** từ Pulsar topics, chứa JSON event từ Vision module

In [4]:
# List all tables
df_tables = query("SHOW TABLES FROM lakehouse.rva")
df_tables

🔍 Query: SHOW TABLES FROM lakehouse.rva


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 12 rows


,Table
0,bronze_raw
1,gold_alert_events
2,gold_alerts
3,gold_camera_daily_dwell
4,gold_camera_daily_metrics
5,gold_camera_hourly_metrics
6,gold_queue_sessions
7,gold_track_summary
8,gold_track_summary_v2
9,gold_zone_minute_metrics


In [5]:
# Describe Bronze table schema
df_schema = query("DESCRIBE lakehouse.rva.bronze_raw")
df_schema

🔍 Query: DESCRIBE lakehouse.rva.bronze_raw


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 8 rows


,Column,Type,Extra,Comment
0,schema_version,varchar,,
1,event_id,varchar,,
2,pipeline_run_id,varchar,,
3,frame_index,bigint,,
4,payload,varchar,,
5,camera_id,varchar,,
6,store_id,varchar,,
7,ingest_ts,timestamp(6),,


In [6]:
# Count total records in Bronze
df_count = query("SELECT COUNT(*) as total_records FROM lakehouse.rva.bronze_raw")
total = df_count.iloc[0]['total_records']
print(f"\n📊 Total Bronze records: {total:,}")
df_count

🔍 Query: SELECT COUNT(*) as total_records FROM lakehouse.rva.bronze_raw


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

📊 Total Bronze records: 2,936


,total_records
0,2936


In [7]:
# Check latest data timestamp
df_latest = query("""
SELECT 
    MIN(ingest_ts) as earliest,
    MAX(ingest_ts) as latest,
    CURRENT_TIMESTAMP as now,
    CAST((CURRENT_TIMESTAMP - MAX(ingest_ts)) AS VARCHAR) as data_lag
FROM lakehouse.rva.bronze_raw
""")
print("\n⏰ Data freshness:")
df_latest

🔍 Query: 
SELECT 
    MIN(ingest_ts) as earliest,
    MAX(ingest_ts) as latest,
    CURRENT_TIMESTAMP as now,...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

⏰ Data freshness:


,earliest,latest,now,data_lag
0,2025-12-06 19:48:59.992,2025-12-06 19:52:21.064,2025-12-06 19:53:04.788000+00:00,0 07:00:43.724


In [6]:
query("""
SELECT
    event_id,
    camera_id,
    store_id,
    frame_index,
    substr(payload, 1, 500) AS payload_sample
FROM lakehouse.rva.bronze_raw
ORDER BY ingest_ts DESC
LIMIT 1
""")

🔍 Query: 
SELECT
    event_id,
    camera_id,
    store_id,
    frame_index,
    substr(payload, 1, 500) AS p...


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,event_id,camera_id,store_id,frame_index,payload_sample
0,24b95ef2a0372499,cam_02,store_001,130,"{""schema_version"":""1.0"",""event_type"":""detection_frame"",""pipeline_run_id"":""387517f7e95b"",""source""..."


In [7]:
query("""
SELECT
    event_id,
    detection_id,
    camera_id,
    frame_index,
    class_name,
    conf,
    track_id,
    global_track_id,
    anchor_x_norm,
    anchor_y_norm,
    primary_zone_id,
    queue_zone_id
FROM lakehouse.rva.silver_detections_v2
LIMIT 10
""")

🔍 Query: 
SELECT
    event_id,
    detection_id,
    camera_id,
    frame_index,
    class_name,
    conf,
  ...


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,event_id,detection_id,camera_id,frame_index,class_name,conf,track_id,global_track_id,anchor_x_norm,anchor_y_norm,primary_zone_id,queue_zone_id
0,267b4414dfa4ccf9,267b4414dfa4ccf9:226-0,cam_02,226,person,0.809814,0,cam_02_g_000000,0.757977,0.300877,None,None
1,267b4414dfa4ccf9,267b4414dfa4ccf9:226-1,cam_02,226,person,0.877441,4,cam_02_g_000004,0.027015,0.699099,None,None
2,267b4414dfa4ccf9,267b4414dfa4ccf9:226-2,cam_02,226,person,0.781576,18,cam_02_g_000018,0.080610,0.534317,aisle_01,None
3,267b4414dfa4ccf9,267b4414dfa4ccf9:226-3,cam_02,226,person,0.897298,20,cam_02_g_000020,0.131619,0.745568,aisle_01,None
4,267b4414dfa4ccf9,267b4414dfa4ccf9:226-4,cam_02,226,person,0.887207,21,cam_02_g_000021,0.409473,0.844444,aisle_01,None
5,267b4414dfa4ccf9,267b4414dfa4ccf9:226-5,cam_02,226,person,0.896484,22,cam_02_g_000022,0.243359,0.395486,aisle_01,None
6,1b4d9d3b0afcd6ef,1b4d9d3b0afcd6ef:222-0,cam_01,222,person,0.504395,62,cam_01_g_000062,0.403866,0.121484,None,None
7,1b4d9d3b0afcd6ef,1b4d9d3b0afcd6ef:222-1,cam_01,222,person,0.307902,96,cam_01_g_000096,0.084352,0.414839,None,None
8,1b4d9d3b0afcd6ef,1b4d9d3b0afcd6ef:222-2,cam_01,222,person,0.464966,97,cam_01_g_000097,0.089816,0.566254,None,None
9,1b4d9d3b0afcd6ef,1b4d9d3b0afcd6ef:222-3,cam_01,222,person,0.469645,99,cam_01_g_000099,0.104548,0.334526,None,None


In [10]:
schema = query("DESCRIBE lakehouse.rva.gold_track_summary")
schema

🔍 Query: DESCRIBE lakehouse.rva.gold_track_summary


C:\spark-tmp\ipykernel_3552\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 9 rows


,Column,Type,Extra,Comment
0,store_id,varchar,,
1,camera_id,varchar,,
2,pipeline_run_id,varchar,,
3,track_id,bigint,,
4,visit_date,date,,
5,enter_ts,timestamp(6) with time zone,,
6,exit_ts,timestamp(6) with time zone,,
7,duration_sec,bigint,,
8,frames,bigint,,


In [9]:
query("""
SELECT
    store_id,
    camera_id,
    global_track_id,
    visit_date,
    enter_ts,
    exit_ts,
    duration_sec,
    frames,
    representative_zone_id
FROM lakehouse.rva.gold_track_summary_v2
LIMIT 10
""")

🔍 Query: 
SELECT
    store_id,
    camera_id,
    global_track_id,
    visit_date,
    enter_ts,
    exit_ts,...


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,store_id,camera_id,global_track_id,visit_date,enter_ts,exit_ts,duration_sec,frames,representative_zone_id
0,store_001,cam_01,cam_01_g_000013,2026-06-12,2026-06-12 03:17:28.720000+00:00,2026-06-12 03:17:32.347000+00:00,3,12,None
1,store_001,cam_01,cam_01_g_000004,2026-06-12,2026-06-12 03:17:26.328000+00:00,2026-06-12 03:17:32.593000+00:00,6,21,checkout_queue_02
2,store_001,cam_01,cam_01_g_000009,2026-06-12,2026-06-12 03:17:27.068000+00:00,2026-06-12 03:17:33.228000+00:00,6,21,checkout_queue_02
3,store_001,cam_02,cam_02_g_000002,2026-06-12,2026-06-12 03:17:26.876000+00:00,2026-06-12 03:17:33.335000+00:00,6,22,aisle_01
4,store_001,cam_01,cam_01_g_000015,2026-06-12,2026-06-12 03:17:29.838000+00:00,2026-06-12 03:17:33.482000+00:00,3,12,None
5,store_001,cam_02,cam_02_g_000001,2026-06-12,2026-06-12 03:17:26.443000+00:00,2026-06-12 03:17:33.639000+00:00,7,24,None
6,store_001,cam_02,cam_02_g_000003,2026-06-12,2026-06-12 03:17:26.876000+00:00,2026-06-12 03:17:34.448000+00:00,7,25,aisle_01
7,store_001,cam_01,cam_01_g_000017,2026-06-12,2026-06-12 03:17:32.084000+00:00,2026-06-12 03:17:35.636000+00:00,3,11,None
8,store_001,cam_01,cam_01_g_000019,2026-06-12,2026-06-12 03:17:33.482000+00:00,2026-06-12 03:17:37.133000+00:00,3,9,None
9,store_001,cam_01,cam_01_g_000011,2026-06-12,2026-06-12 03:17:27.914000+00:00,2026-06-12 03:17:37.544000+00:00,9,28,None


In [9]:
df_bronze_sample = query("""
SELECT
    store_id,
    camera_id,
    frame_index,
    ingest_ts,
    json_extract_scalar(payload, '$.frame_index') AS frame_index_json
FROM lakehouse.rva.bronze_raw
ORDER BY ingest_ts DESC
LIMIT 10
""")
df_bronze_sample


🔍 Query: 
SELECT
    store_id,
    camera_id,
    frame_index,
    ingest_ts,
    json_extract_scalar(payload...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,store_id,camera_id,frame_index,ingest_ts,frame_index_json
0,store_01,cam_01,2936,2025-12-06 19:52:21.064,2936
1,store_01,cam_01,2935,2025-12-06 19:52:21.063,2935
2,store_01,cam_01,2934,2025-12-06 19:52:21.063,2934
3,store_01,cam_01,2933,2025-12-06 19:52:21.062,2933
4,store_01,cam_01,2931,2025-12-06 19:52:21.061,2931
5,store_01,cam_01,2932,2025-12-06 19:52:21.061,2932
6,store_01,cam_01,2930,2025-12-06 19:52:21.060,2930
7,store_01,cam_01,2929,2025-12-06 19:52:21.060,2929
8,store_01,cam_01,2927,2025-12-06 19:52:21.059,2927
9,store_01,cam_01,2928,2025-12-06 19:52:21.059,2928


In [10]:
tables = [
    "bronze_raw",
    "silver_detections",
    "silver_detections_v2",
    "gold_track_summary",
    "gold_track_summary_v2",
    "gold_camera_hourly_metrics",
    "gold_camera_daily_metrics",
    "gold_camera_daily_dwell",
    "gold_queue_sessions",
    "gold_zone_minute_metrics",
    "gold_alert_events",
    "gold_alerts",
]

rows = []

for table in tables:
    try:
        count = query(f"SELECT COUNT(*) AS row_count FROM lakehouse.rva.{table}").iloc[0, 0]
        rows.append({"table": table, "row_count": count})
    except Exception as e:
        rows.append({"table": table, "row_count": "not available"})

pd.DataFrame(rows)

🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.bronze_raw


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.silver_detections
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.silver_detections_v2
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_track_summary
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_track_summary_v2
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_camera_hourly_metrics
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_camera_daily_metrics
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_camera_daily_dwell
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_queue_sessions
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_zone_minute_metrics
✅ Returned 1 rows
🔍 Query: SELECT COUNT(*) AS row_count FROM lakehouse.rva.gold_alert_events
✅ Returned 1 rows
🔍 Query: SEL

,table,row_count
0,bronze_raw,709
1,silver_detections,6089
2,silver_detections_v2,7674
3,gold_track_summary,244
4,gold_track_summary_v2,256
5,gold_camera_hourly_metrics,0
6,gold_camera_daily_metrics,0
7,gold_camera_daily_dwell,0
8,gold_queue_sessions,60
9,gold_zone_minute_metrics,0


In [9]:
schema = query("DESCRIBE lakehouse.rva.silver_detections")
schema

🔍 Query: DESCRIBE lakehouse.rva.silver_detections


C:\spark-tmp\ipykernel_3552\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 20 rows


,Column,Type,Extra,Comment
0,schema_version,varchar,,
1,event_id,varchar,,
2,detection_id,varchar,,
3,pipeline_run_id,varchar,,
4,store_id,varchar,,
5,camera_id,varchar,,
6,frame_index,bigint,,
7,capture_ts,timestamp(6),,
8,img_w,integer,,
9,img_h,integer,,


In [10]:
df_frame_stats = query("""
SELECT
    COUNT(*)                                     AS total_records,
    COUNT(*) FILTER (WHERE frame_index IS NULL)  AS null_frame_index,
    COUNT(*) FILTER (WHERE frame_index IS NOT NULL) AS non_null_frame_index
FROM lakehouse.rva.bronze_raw
""")
df_frame_stats


🔍 Query: 
SELECT
    COUNT(*)                                     AS total_records,
    COUNT(*) FILTER (WHER...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,total_records,null_frame_index,non_null_frame_index
0,2936,0,2936


In [11]:
bronze = query("""
SELECT
    event_id,
    pipeline_run_id,
    camera_id,
    store_id,
    frame_index,
    ingest_ts,
    payload
FROM lakehouse.rva.bronze_raw
ORDER BY ingest_ts DESC
LIMIT 1
""")

bronze

🔍 Query: 
SELECT
    event_id,
    pipeline_run_id,
    camera_id,
    store_id,
    frame_index,
    ingest_...


C:\spark-tmp\ipykernel_5532\638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,event_id,pipeline_run_id,camera_id,store_id,frame_index,ingest_ts,payload
0,24b95ef2a0372499,387517f7e95b,cam_02,store_001,130,2026-06-12 03:19:09.262,"{""schema_version"":""1.0"",""event_type"":""detection_frame"",""pipeline_run_id"":""387517f7e95b"",""source""..."


In [11]:
# View one full raw JSON event
df_raw = query("""
SELECT payload 
FROM lakehouse.rva.bronze_raw 
ORDER BY ingest_ts DESC 
LIMIT 1
""")

if df_raw is not None and len(df_raw) > 0:
    raw_json = json.loads(df_raw.iloc[0]['payload'])
    print("\n📄 Sample Bronze event structure:")
    print(json.dumps(raw_json, indent=2)[:1500])
    print("\n[...output truncated...]")

🔍 Query: 
SELECT payload 
FROM lakehouse.rva.bronze_raw 
ORDER BY ingest_ts DESC 
LIMIT 1



/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

📄 Sample Bronze event structure:
{
  "schema_version": "1.0",
  "pipeline_run_id": "efb8514d238549ff9892e543f3cc30bd",
  "source": {
    "store_id": "store_01",
    "camera_id": "cam_01",
    "stream_id": "stream_01"
  },
  "frame_index": 2936,
  "capture_ts": "2025-12-06T19:52:21.000781+00:00",
  "image_size": {
    "width": 1920,
    "height": 1080
  },
  "detections": [
    {
      "det_id": "2936-0",
      "class": "person",
      "class_id": 0,
      "conf": 0.8762192130088806,
      "bbox": {
        "x1": 1213.7457275390625,
        "y1": 323.0450134277344,
        "x2": 1419.415771484375,
        "y2": 828.622802734375
      },
      "bbox_norm": {
        "x": 0.6321592330932617,
        "y": 0.2991157531738281,
        "w": 0.10711981455485026,
        "h": 0.4681275826913339
      },
      "centroid": {
        "x": 1316,
        "y": 575
      },
      "centroid_norm": {
        "x": 0.6857191403706868,
        "y": 0.5331795445194951
      },
      "trac

### 📊 Bronze Data Quality Checks

In [12]:
# Check for NULL values
df_nulls = query("""
SELECT 
    COUNT(*) as total_records,
    COUNT(*) FILTER (WHERE payload IS NULL) as null_payload,
    COUNT(*) FILTER (WHERE store_id IS NULL) as null_store_id,
    COUNT(*) FILTER (WHERE camera_id IS NULL) as null_camera_id
FROM lakehouse.rva.bronze_raw
""")
print("\n🔍 NULL check:")
df_nulls

🔍 Query: 
SELECT 
    COUNT(*) as total_records,
    COUNT(*) FILTER (WHERE payload IS NULL) as null_payload,...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows

🔍 NULL check:


,total_records,null_payload,null_store_id,null_camera_id
0,2936,0,0,0


In [13]:
df_silver_stats = query("""
SELECT
    COUNT(*)                               AS total_rows,
    COUNT(DISTINCT store_id)               AS stores,
    COUNT(DISTINCT camera_id)              AS cameras,
    COUNT(DISTINCT pipeline_run_id)        AS pipeline_runs,
    COUNT(DISTINCT track_id)               AS distinct_tracks,
    MIN(capture_ts)                        AS earliest_capture,
    MAX(capture_ts)                        AS latest_capture
FROM lakehouse.rva.silver_detections
""")
df_silver_stats


🔍 Query: 
SELECT
    COUNT(*)                               AS total_rows,
    COUNT(DISTINCT store_id)      ...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,total_rows,stores,cameras,pipeline_runs,distinct_tracks,earliest_capture,latest_capture
0,19021,1,1,1,146,2025-12-06 19:48:56.216,2025-12-06 19:52:21


In [14]:
df_silver_sample = query("""
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    det_id,
    frame_index,
    capture_ts,
    processing_ts,
    class_name,
    class_id,
    conf,
    bbox_x1,
    bbox_y1,
    bbox_x2,
    bbox_y2
FROM lakehouse.rva.silver_detections
ORDER BY capture_ts DESC
LIMIT 10
""")
df_silver_sample


🔍 Query: 
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    det_id,
    frame_index,...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,store_id,camera_id,pipeline_run_id,track_id,det_id,frame_index,capture_ts,processing_ts,class_name,class_id,conf,bbox_x1,bbox_y1,bbox_x2,bbox_y2
0,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,167,2936-3,2936,2025-12-06 19:52:21.000,2025-12-06 19:52:26.499000+00:00,person,0,0.729937,952,48,1014,205
1,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,22,2936-0,2936,2025-12-06 19:52:21.000,2025-12-06 19:52:26.499000+00:00,person,0,0.876219,1214,323,1419,829
2,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,171,2936-4,2936,2025-12-06 19:52:21.000,2025-12-06 19:52:26.499000+00:00,person,0,0.599186,739,5,787,114
3,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,157,2936-2,2936,2025-12-06 19:52:21.000,2025-12-06 19:52:26.499000+00:00,person,0,0.590177,1187,265,1327,682
4,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,172,2936-6,2936,2025-12-06 19:52:21.000,2025-12-06 19:52:26.499000+00:00,person,0,0.485887,1197,266,1327,532
5,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,167,2935-3,2935,2025-12-06 19:52:20.925,2025-12-06 19:52:26.498000+00:00,person,0,0.731332,952,48,1014,205
6,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,155,2935-1,2935,2025-12-06 19:52:20.925,2025-12-06 19:52:26.498000+00:00,person,0,0.771652,1181,170,1281,416
7,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,22,2935-0,2935,2025-12-06 19:52:20.925,2025-12-06 19:52:26.497000+00:00,person,0,0.878114,1214,323,1420,829
8,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,157,2935-2,2935,2025-12-06 19:52:20.925,2025-12-06 19:52:26.498000+00:00,person,0,0.589799,1189,264,1327,682
9,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,171,2935-4,2935,2025-12-06 19:52:20.925,2025-12-06 19:52:26.498000+00:00,person,0,0.624421,739,5,788,113


In [15]:
df_conf = query("""
SELECT 
    approx_percentile(conf, 0.5) AS median_conf,
    approx_percentile(conf, 0.9) AS p90_conf,
    approx_percentile(conf, 0.99) AS p99_conf,
    COUNT(*) AS total_detections
FROM lakehouse.rva.silver_detections
""")
df_conf


🔍 Query: 
SELECT 
    approx_percentile(conf, 0.5) AS median_conf,
    approx_percentile(conf, 0.9) AS p90_co...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,median_conf,p90_conf,p99_conf,total_detections
0,0.729113,0.902296,0.931496,19021


In [16]:
df_per_frame = query("""
SELECT 
    frame_index,
    COUNT(*) AS detections
FROM lakehouse.rva.silver_detections
GROUP BY frame_index
ORDER BY detections DESC
LIMIT 10
""")
df_per_frame


🔍 Query: 
SELECT 
    frame_index,
    COUNT(*) AS detections
FROM lakehouse.rva.silver_detections
GROUP BY f...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,frame_index,detections
0,1975,12
1,2002,11
2,1976,11
3,1977,11
4,243,10
5,93,10
6,1968,10
7,1984,10
8,2007,10
9,1969,10


In [17]:
query("""
SELECT COUNT(*) AS total_tracks
FROM lakehouse.rva.gold_track_summary
""")


🔍 Query: 
SELECT COUNT(*) AS total_tracks
FROM lakehouse.rva.gold_track_summary



/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,total_tracks
0,102


In [18]:
query("DESCRIBE lakehouse.rva.gold_track_summary")

🔍 Query: DESCRIBE lakehouse.rva.gold_track_summary


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 9 rows


,Column,Type,Extra,Comment
0,store_id,varchar,,
1,camera_id,varchar,,
2,pipeline_run_id,varchar,,
3,track_id,bigint,,
4,visit_date,date,,
5,enter_ts,timestamp(6) with time zone,,
6,exit_ts,timestamp(6) with time zone,,
7,duration_sec,bigint,,
8,frames,bigint,,


In [19]:
df_gold_sample = query("""
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    visit_date,
    enter_ts,
    exit_ts,
    duration_sec,
    frames
FROM lakehouse.rva.gold_track_summary
ORDER BY exit_ts DESC
LIMIT 10
""")
df_gold_sample


🔍 Query: 
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    visit_date,
    enter_ts...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,store_id,camera_id,pipeline_run_id,track_id,visit_date,enter_ts,exit_ts,duration_sec,frames
0,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,63,2025-12-06,2025-12-06 19:50:13.887000+00:00,2025-12-06 19:51:20.654000+00:00,66,894
1,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,113,2025-12-06,2025-12-06 19:51:08.754000+00:00,2025-12-06 19:51:20.654000+00:00,11,177
2,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,22,2025-12-06,2025-12-06 19:49:13.414000+00:00,2025-12-06 19:51:20.654000+00:00,127,1649
3,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,111,2025-12-06,2025-12-06 19:51:07.880000+00:00,2025-12-06 19:51:20.654000+00:00,12,190
4,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,102,2025-12-06,2025-12-06 19:50:50.870000+00:00,2025-12-06 19:51:20.654000+00:00,29,415
5,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,86,2025-12-06,2025-12-06 19:50:37.316000+00:00,2025-12-06 19:51:20.654000+00:00,43,626
6,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,116,2025-12-06,2025-12-06 19:51:13.410000+00:00,2025-12-06 19:51:20.530000+00:00,7,31
7,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,106,2025-12-06,2025-12-06 19:51:01.293000+00:00,2025-12-06 19:51:20.094000+00:00,18,199
8,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,120,2025-12-06,2025-12-06 19:51:15.892000+00:00,2025-12-06 19:51:19.523000+00:00,3,54
9,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,121,2025-12-06,2025-12-06 19:51:17.163000+00:00,2025-12-06 19:51:19.251000+00:00,2,18


In [20]:
df_gold_stats = query("""
SELECT
    COUNT(*)              AS total_tracks,
    AVG(duration_sec)     AS avg_duration_sec,
    MIN(duration_sec)     AS min_duration_sec,
    MAX(duration_sec)     AS max_duration_sec,
    AVG(frames)           AS avg_frames
FROM lakehouse.rva.gold_track_summary
""")
df_gold_stats


🔍 Query: 
SELECT
    COUNT(*)              AS total_tracks,
    AVG(duration_sec)     AS avg_duration_sec,
  ...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,total_tracks,avg_duration_sec,min_duration_sec,max_duration_sec,avg_frames
0,102,9.990196,0,127,127.382353


In [23]:
# Schema + tổng số dòng Gold
df_gold_schema = query("DESCRIBE lakehouse.rva.gold_track_summary")
df_gold_schema

df_gold_rows = query("""
SELECT COUNT(*) AS total_rows 
FROM lakehouse.rva.gold_track_summary
""")
df_gold_rows


🔍 Query: DESCRIBE lakehouse.rva.gold_track_summary


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 9 rows
🔍 Query: 
SELECT COUNT(*) AS total_rows 
FROM lakehouse.rva.gold_track_summary

✅ Returned 1 rows


,total_rows
0,102


In [26]:
df_gold_keys = query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) AS distinct_keys
FROM (
    SELECT
        store_id,
        camera_id,
        pipeline_run_id,
        track_id
    FROM lakehouse.rva.gold_track_summary
    GROUP BY
        store_id,
        camera_id,
        pipeline_run_id,
        track_id
) t
""")
df_gold_keys


🔍 Query: 
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) AS distinct_keys
FROM (
    SELECT
        store_id...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 1 rows


,total_rows,distinct_keys
0,378,378


In [25]:
df_gold_latest = query("""
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    visit_date,
    enter_ts,
    exit_ts,
    duration_sec,
    frames
FROM lakehouse.rva.gold_track_summary
ORDER BY exit_ts DESC
LIMIT 10
""")
df_gold_latest


🔍 Query: 
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    visit_date,
    enter_ts...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 10 rows


,store_id,camera_id,pipeline_run_id,track_id,visit_date,enter_ts,exit_ts,duration_sec,frames
0,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,172,2025-12-06,2025-12-06 19:52:20.547000+00:00,2025-12-06 19:52:21+00:00,0,7
1,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,22,2025-12-06,2025-12-06 19:49:13.414000+00:00,2025-12-06 19:52:21+00:00,187,2540
2,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,167,2025-12-06,2025-12-06 19:52:17.021000+00:00,2025-12-06 19:52:21+00:00,3,59
3,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,157,2025-12-06,2025-12-06 19:52:07.211000+00:00,2025-12-06 19:52:21+00:00,13,210
4,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,171,2025-12-06,2025-12-06 19:52:19.980000+00:00,2025-12-06 19:52:21+00:00,1,16
5,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,155,2025-12-06,2025-12-06 19:52:02.744000+00:00,2025-12-06 19:52:20.925000+00:00,18,277
6,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,169,2025-12-06,2025-12-06 19:52:18.519000+00:00,2025-12-06 19:52:20.608000+00:00,2,13
7,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,170,2025-12-06,2025-12-06 19:52:19.411000+00:00,2025-12-06 19:52:20.265000+00:00,0,12
8,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,154,2025-12-06,2025-12-06 19:52:02.615000+00:00,2025-12-06 19:52:18.391000+00:00,15,215
9,store_01,cam_01,efb8514d238549ff9892e543f3cc30bd,164,2025-12-06,2025-12-06 19:52:13.548000+00:00,2025-12-06 19:52:16.380000+00:00,2,45


In [27]:
df_gold_dups = query("""
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    COUNT(*) AS cnt
FROM lakehouse.rva.gold_track_summary
GROUP BY
    store_id,
    camera_id,
    pipeline_run_id,
    track_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20
""")
df_gold_dups

🔍 Query: 
SELECT
    store_id,
    camera_id,
    pipeline_run_id,
    track_id,
    COUNT(*) AS cnt
FROM lak...


/tmp/ipykernel_327198/638732029.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


✅ Returned 0 rows


,store_id,camera_id,pipeline_run_id,track_id,cnt
